# Agents with smolagents

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

So far in this course we have seen how to **prompt** an LLM (single turn) and how to **augment** it with retrieved context (RAG). Both approaches share a common limitation: the model receives an input, runs one forward pass, and produces a single output. It cannot loop, backtrack, or use tools dynamically.

**Agents** remove this limitation. An agent is a system in which the LLM acts as a reasoning engine that decides, step by step, what to do next — querying tools, inspecting results, and repeating until the task is complete. This is the architecture behind modern AI assistants that browse the web, write and execute code, and orchestrate multi-step workflows.

We use **[smolagents](https://github.com/huggingface/smolagents)** — a lightweight HuggingFace library that provides two agent types, a rich set of built-in tools, and clean primitives for building custom tools and multi-agent pipelines. Crucially, everything in this notebook runs **locally** using a quantised model.

We cover:
1. What is an agent? (observe → think → act)
2. Setup: smolagents, imports, and local model loading
3. Your first agent — a bare `CodeAgent` with no tools
4. Built-in tools (`PythonInterpreterTool`, `FinalAnswerTool`)
5. Writing a custom tool with the `@tool` decorator
6. Tool composition — chaining multiple tools in one task
7. `CodeAgent` vs `ToolCallingAgent` — comparing internal reasoning
8. Inspecting the agent's memory
9. Multi-step reasoning — tasks requiring sequential steps
10. Controlling the agent — `max_steps` and graceful failure
11. Advanced: multi-agent systems with `managed_agents`

## 1  What is an Agent?

A **plain LLM call** maps one prompt to one response. An **agent** wraps that LLM in a loop:

```
User query
    │
    ▼
┌───────────────────────────────┐
│  OBSERVE — read task / tool   │
│           result              │
│                               │
│  THINK   — LLM reasons about  │
│           what to do next     │
│                               │
│  ACT     — call a tool, write │
│           code, or answer     │
└───────────┬───────────────────┘
            │ repeat until done
            ▼
      Final answer
```

### The three parts of an agentic system

| Component | Role |
|---|---|
| **LLM (brain)** | Reads context, reasons, decides the next action |
| **Tools** | External functions the LLM can call (calculator, search, code runner, …) |
| **Executor / loop** | Runs the tool, feeds results back to the LLM, detects when the task is done |

### Agent vs plain LLM call

| | Plain LLM | Agent |
|---|---|---|
| Steps | 1 (prompt → response) | N (loop until done) |
| Memory | None (stateless) | Step log keeps all prior observations |
| Tools | None | Any callable function |
| Self-correction | Impossible | Can retry after an error |

The ReAct paper (Yao et al., 2022) showed that interleaving **reasoning** ("Thought") with **actions** (tool calls) dramatically improves performance on tasks that require planning or factual look-ups. Modern agent frameworks — including smolagents — implement exactly this pattern under the hood.

## 2  Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `smolagents` | Agent framework: `CodeAgent`, `ToolCallingAgent`, built-in tools, `@tool` decorator |
| `transformers` | Local model loading, tokenisers, pipelines |
| `bitsandbytes` (optional)| 4-bit quantisation — reduces VRAM from ~3 GB to ~1 GB for a 1.5 B model |
| `accelerate` | Device placement and mixed-precision support |

In [ ]:
!pip install -q -U smolagents transformers bitsandbytes accelerate

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import BitsAndBytesConfig

from smolagents import (
    CodeAgent,
    ToolCallingAgent,
    TransformersModel,
    PythonInterpreterTool,
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    tool,
)
from smolagents.memory import ActionStep, FinalAnswerStep, SystemPromptStep, TaskStep

## 3  Loading the Model

We use **Qwen** models throughout this notebook. Some of them are small enough to run on a consumer GPU (or even CPU), yet capable enough to follow multi-step agentic instructions.

In [2]:
MODEL_ID = "Qwen/Qwen3.5-4B"

model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",                          # Distribute across available GPUs automatically
    max_new_tokens=8096,
)

print(f"Model loaded: {MODEL_ID}")

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:59<00:00, 12.12it/s]


Model loaded: Qwen/Qwen3.5-4B


`TransformersModel` wraps a HuggingFace model in the `smolagents` `Model` interface, handling chat-template formatting and tool-call parsing automatically. We load the model **once here** and pass it to every agent we create below.

## 4  Your First Agent

The simplest possible agent: `CodeAgent(tools=[], model=model)`. `tools=[]` does **not** mean the agent is powerless — it means there are no pre-wrapped smolagents tools. The agent still has full access to the **Python executor**: it can import any installed package, call external APIs, manipulate files, and run arbitrary computations. The executor *is* the capability.

`CodeAgent` works by asking the LLM to write **Python code** that solves each step. The executor runs that code and feeds the output back to the LLM as the next observation — making the entire Python ecosystem available as a free-form tool.

In [3]:
agent = CodeAgent(
    tools=[],
    model=model,
    code_block_tags="markdown",
    executor_type="local",
    max_steps=5,
    stream_outputs=True
)

print(agent.system_prompt)

result = agent.run(
    "If a snail travels 3.7 cm per minute, how many minutes does it need "
    "to cross a 2.5 m garden path? Round to the nearest whole minute."
)
print("\nFinal answer:", result)


You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '```python', and closed with '```'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples u

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If a snail travels 3.7 cm per minute, how many minutes does it need to cross a 2.5 m garden path? Round to the  │
│ nearest whole minute.                                                                                           │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Convert distance from meters to centimeters                                                                    
  distance_m = 2.5                                                                                                 
  distance_cm = distance_m * 100                                                                                   
                                                                                                                   
  # Snail speed in cm per minute                                                                                   
  speed_cm_per_min = 3.7                                                                                           
                                                                                                                   
  # Calculate time in minutes                                                                                      
  time_minutes = distance_cm / speed_cm_per_min                                                                    
                                                                                                                   
  # Round to nearest whole minute                                                                                  
  time_rounded = round(time_minutes)                                                                               
                                                                                                                   
  print(f"Distance: {distance_cm} cm")                                                                             
  print(f"Speed: {speed_cm_per_min} cm/min")                                                                       
  print(f"Time: {time_minutes} minutes")                                                                           
  print(f"Rounded time: {time_rounded} minutes")                                                                   
                                                                                                                   
  final_answer(time_rounded)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Distance: 250.0 cm
Speed: 3.7 cm/min
Time: 67.56756756756756 minutes
Rounded time: 68 minutes

Final answer: 68

[Step 1: Duration 24.56 seconds| Input tokens: 2,131 | Output tokens: 406]


Final answer: 68


The agent's reasoning trace appears in the output above. Notice the structure:

1. **System prompt step** — smolagents injects the tool list and instructions.
2. **Task step** — your question is added to the conversation.
3. **Action step(s)** — the LLM emits a code block; the executor runs it; the result becomes the observation.
4. **Final answer step** — the LLM calls `final_answer(...)` to signal it is done.

Even with `tools=[]` the agent wrote valid Python to perform the unit conversion and returned the correct answer.

> **How `CodeAgent` works under the hood**
> 
> At each step the LLM is asked to produce a Python code block. The local executor runs that block in a sandboxed namespace, captures `print` output and the return value, and returns them as the next observation. The LLM can therefore use any Python expression — arithmetic, string formatting, data structures — as a free-form calculator without relying on dedicated tools.

## 5  Built-in Tools

smolagents ships with a set of ready-made tools. The most commonly used ones:

| Tool class | What it does |
|---|---|
| `PythonInterpreterTool` | Runs arbitrary Python code (explicit tool-call flavour) |
| `FinalAnswerTool` | Signals the end of the run and sets the return value |
| `DuckDuckGoSearchTool` | Web search via DuckDuckGo (requires internet access) |
| `WikipediaSearchTool` | Searches Wikipedia articles |
| `VisitWebpageTool` | Fetches and parses a URL |

Tools are passed as a **list** to the agent constructor. The model reads each tool's name and docstring to decide when to call it — so **tool descriptions matter** (see callout below).

Let's demonstrate `PythonInterpreterTool` and `FinalAnswerTool` on a task that requires real computation:

In [4]:
agent_tools = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=6,
    additional_authorized_imports=["numpy"],   # Agent may import numpy in its generated code
)

result = agent_tools.run(
    "Compute the sum of all prime numbers less than 50. Use numpy module."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Compute the sum of all prime numbers less than 50. Use numpy module.                                            │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import numpy as np                                                                                               
                                                                                                                   
  # Create array of numbers from 0 to 49                                                                           
  numbers = np.arange(50)                                                                                          
                                                                                                                   
  # Function to check if a number is prime                                                                         
  def is_prime(n):                                                                                                 
      if n < 2:                                                                                                    
          return False                                                                                             
      if n == 2:                                                                                                   
          return True                                                                                              
      if n % 2 == 0:                                                                                               
          return False                                                                                             
      for i in range(3, int(np.sqrt(n)) + 1, 2):                                                                   
          if n % i == 0:                                                                                           
              return False                                                                                         
      return True                                                                                                  
                                                                                                                   
  # Find all prime numbers less than 50                                                                            
  primes = [n for n in numbers if is_prime(n)]                                                                     
                                                                                                                   
  # Sum the primes                                                                                                 
  prime_sum = sum(primes)                                                                                          
                                                                                                                   
  print(f"Primes less than 50: {primes}")                                                                          
  print(f"Sum: {prime_sum}")                                                                                       
                                                                                                                   
  final_answer(prime_sum)                                                                                          
                                                                                                                   
  import numpy as np                                                                                               
                                                                                                                   
  # Create array of numbers from 0 to 49                                                                           
  numbers = np.arange(50)                                                                                          
                                                         

Execution logs:
Primes less than 50: [np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(11), np.int64(13), np.int64(17),
np.int64(19), np.int64(23), np.int64(29), np.int64(31), np.int64(37), np.int64(41), np.int64(43), np.int64(47)]
Sum: 328

Final answer: 328

[Step 1: Duration 51.52 seconds| Input tokens: 2,230 | Output tokens: 894]


Final answer: 328


With `PythonInterpreterTool` available the agent can write and execute code snippets as explicit **tool calls** rather than inline scratchpad code. `FinalAnswerTool` gives it a clean way to commit to a final value.

> **Why tool descriptions matter**
>
> The LLM never sees the tool's source code — only its **name** and **docstring**. These two pieces of text are what the model uses to decide whether to call a tool and how to pass arguments. A vague docstring like `"Does something with text"` will cause the model to ignore the tool or misuse it. A precise description like `"Counts words, sentences, and characters in a text string"` makes the tool reliably discoverable.

## 6  Writing a Custom Tool

Any Python function decorated with `@tool` becomes a first-class smolagents tool. The decorator introspects the function's **name**, **docstring**, and **type annotations** to build the tool description that the model reads.

Rules for a well-formed tool:
- All parameters must have type annotations.
- The docstring must describe what the tool does *and* document each argument (NumPy or Google style).
- The return type should be one of `str`, `int`, `float`, `bool`, or `dict`.

In [5]:
@tool
def text_statistics(text: str) -> dict:
    """Compute basic statistics for a piece of text.

    Returns a dictionary with:
    - word_count (int): number of words
    - sentence_count (int): number of sentences
    - avg_word_length (float): average word length
    - char_count (int): total character count excluding whitespace

    Args:
        text: The input text to analyse.
    """
    words = text.split()
    sentences = max(1, text.count(".") + text.count("!") + text.count("?"))
    avg_word_len = sum(len(w.strip(".,!?")) for w in words) / max(1, len(words))
    return {
        "word_count": len(words),
        "sentence_count": sentences,
        "avg_word_length": round(avg_word_len, 2),
        "char_count": sum(len(w) for w in words),
    }


# Inspect what the model will see
print("Tool name  :", text_statistics.name)
print("Description:", text_statistics.description)
print("Inputs     :", text_statistics.inputs)
print("Output type:", text_statistics.output_type)

Tool name  : text_statistics
Description: Compute basic statistics for a piece of text.

Returns a dictionary with:
- word_count (int): number of words
- sentence_count (int): number of sentences
- avg_word_length (float): average word length
- char_count (int): total character count excluding whitespace
Inputs     : {'text': {'type': 'string', 'description': 'The input text to analyse.'}}
Output type: object


In [6]:
agent_custom = CodeAgent(
    tools=[text_statistics, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=5,
)

passage = (
    "The transformer architecture was introduced in 2017. "
    "It relies on a self-attention mechanism that computes relationships "
    "between all tokens in a sequence simultaneously. "
    "This parallelism made training much faster than previous RNN approaches."
)

result = agent_custom.run(
    f"Use the text_statistics tool to analyse the following passage, then "
    f"report the word count and the average word length:\n\n{passage}"
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the text_statistics tool to analyse the following passage, then report the word count and the average word  │
│ length:                                                                                                         │
│                                                                                                                 │
│ The transformer architecture was introduced in 2017. It relies on a self-attention mechanism that computes      │
│ relationships between all tokens in a sequence simultaneously. This parallelism made training much faster than  │
│ previous RNN approaches.                                                                                        │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "The transformer architecture was introduced in 2017. It relies on a self-attention mechanism that        
  computes relationships between all tokens in a sequence simultaneously. This parallelism made training much      
  faster than previous RNN approaches."                                                                            
                                                                                                                   
  stats = text_statistics(text=text)                                                                               
  print(stats)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'word_count': 33, 'sentence_count': 3, 'avg_word_length': 6.27, 'char_count': 210}

Out: None

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  word_count = 33                                                                                                  
  avg_word_length = 6.27                                                                                           
  final_answer(f"Word count: {word_count}, Average word length: {avg_word_length}")                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Word count: 33, Average word length: 6.27

[Step 2: Duration 11.99 seconds| Input tokens: 4,780 | Output tokens: 445]


Final answer: Word count: 33, Average word length: 6.27


The `@tool` decorator requires no boilerplate class definition — a plain annotated function is sufficient. The agent correctly identified `text_statistics` as the right tool, called it with the passage as argument, and summarised the returned dictionary.

## 7  Tool Composition

The real power of agents emerges when they must **chain multiple tools** in sequence. The agent decides on its own which tool to call at each step, in what order, and how to pass intermediate results forward.

We give the agent three tools and pose a task that requires at least two of them in a specific order:

In [7]:
@tool
def word_frequency(text: str, top_n: int) -> dict:
    """Return the top N most frequent words in a text, ignoring case.

    Args:
        text: The input text to analyse.
        top_n: How many top words to return.
    """
    from collections import Counter
    tokens = [w.strip(".,!?;:\"\'()").lower() for w in text.split()]
    tokens = [t for t in tokens if len(t) > 2]   # Drop very short words
    return dict(Counter(tokens).most_common(top_n))


agent_composed = CodeAgent(
    tools=[text_statistics, word_frequency, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=8,
)

corpus = (
    "Natural language processing enables computers to understand human language. "
    "Language models learn statistical patterns from large language corpora. "
    "Understanding language structure is fundamental to natural language processing."
)

result = agent_composed.run(
    "First compute the text statistics for the corpus below, then find the top 3 "
    "most frequent words. Report both results.\n\n" + corpus
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ First compute the text statistics for the corpus below, then find the top 3 most frequent words. Report both    │
│ results.                                                                                                        │
│                                                                                                                 │
│ Natural language processing enables computers to understand human language. Language models learn statistical   │
│ patterns from large language corpora. Understanding language structure is fundamental to natural language       │
│ processing.                                                                                                     │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  corpus = "Natural language processing enables computers to understand human language. Language models learn      
  statistical patterns from large language corpora. Understanding language structure is fundamental to natural     
  language processing."                                                                                            
                                                                                                                   
  text_stats = text_statistics(text=corpus)                                                                        
  print("Text Statistics:", text_stats)                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Text Statistics: {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}

Out: None

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text_stats = {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}                 
  top_words = {'language': 6, 'natural': 2, 'processing': 2}                                                       
  final_answer(f"Text Statistics: {text_stats}\nTop 3 most frequent words: {top_words}")                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Text Statistics: {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}
Top 3 most frequent words: {'language': 6, 'natural': 2, 'processing': 2}

[Step 3: Duration 11.68 seconds| Input tokens: 7,634 | Output tokens: 562]


Final answer: Text Statistics: {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}
Top 3 most frequent words: {'language': 6, 'natural': 2, 'processing': 2}


The tool-call log shows the agent's execution order. It called `text_statistics` first, read the output, then called `word_frequency` — exactly the chain required by the task. The agent inferred this ordering from the task description alone, without any explicit sequencing instructions.

## 8  Web Search with `DuckDuckGoSearchTool`

One of the most useful built-in tools is `DuckDuckGoSearchTool`, which lets the agent query the web at runtime — grounding its answers in live information rather than model weights.

| Property | Detail |
|---|---|
| Class | `smolagents.DuckDuckGoSearchTool` |
| Requires | Internet access; no API key needed |
| Returns | A string with the top search results (titles + snippets) |
| Best for | Factual look-ups, current events, anything that may postdate the model |

You can call the tool directly to inspect its output before involving an agent:


In [8]:
search = DuckDuckGoSearchTool()

# Call the tool directly — useful for debugging what the agent will see
results = search("smolagents HuggingFace library")
print(results[:800])


## Search Results

[GitHub - huggingface/smolagents: 🤗 smolagents: a barebones](https://github.com/huggingface/smolagents)
Misc {smolagents, title = { `smolagents`: a smollibraryto build great agentic systems. ... url{https://github.com/huggingface/smolagents...

[huggingface/smolagents - Gitstar Ranking](https://gitstar-ranking.com/huggingface/smolagents)
huggingface/smolagents...smolagents: a bareboneslibraryfor agents that think in code. ... https://huggingface.co/docs/smolagents

[Hugging Face's Smolagents: A Guide With Examples | DataCamp](https://www.datacamp.com/tutorial/smolagents)
Hugging Face'ssmolagentsis a Pythonlibrarythat simplifies the creation of AI agents , making them more accessible to developers.

[Smolagents Alternatives and Reviews](https://www.libhunt.com/r/smolagent


### 8.1  Agent with web search

Now we hand the tool to a `CodeAgent`. The agent decides *when* to search and *what* to search for — we only specify the goal:


In [13]:
agent_search = CodeAgent(
    tools=[DuckDuckGoSearchTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=5,
)

result = agent_search.run(
    "What is the latest stable version of the smolagents library? "
    "Search for it and report the version number."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest stable version of the smolagents library? Search for it and report the version number.       │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  version_info = web_search(query="smolagents library latest stable version")                                      
  print(version_info)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Stable Diffusion 3: Guide to the Text-to-Image Model 
by](https://www.analyticsvidhya.com/blog/2024/06/stable-diffusion-3/)
StableDiffusion 3: Guide to theLatestText-to-Image Model by Stab... ...StableDiffusion 3 leverages an advanced 
Multimodal Diffusion Transformer ...

[Troubleshooting](https://huggingface.co/docs/peft/en/developer_guides/troubleshooting)
Starting from PEFTversionv0.12.0, PEFT automatically promotes the dtype of adapter weights from torch.float16 and 
torch.bfloat16 to torch.float32 ...

[MetaCLIP 2](https://huggingface.co/docs/transformers/main/model_doc/metaclip_2)
You are viewing mainversion, which requires installation ... If you d like regular pip install, checkout 
thelateststableversion( v5.0.0rc1 ).

[OpenEnv Integration for Training LLMs with Environments](https://huggingface.co/docs/trl/main/en/openenv)
You are viewing mainversion, which requires installation from ... If you d like regular pip install, checkout 
thelateststableversion( v0.26.2 ).

[Custom models](https://huggingface.co/docs/peft/main/en/developer_guides/custom_models)
If you d like regular pip install, checkout thelateststableversion( v0.18.0 ). ... perceptron, a computer vision 
model from the timmlibrary, or ...

[CPO Trainer](https://huggingface.co/docs/trl/main/en/cpo_trainer)
If you d like regular pip install, checkout thelateststableversion( v0.24.0 ). ... the combined use of CPO and 
SimPO, which enables morestable...

[GitHub - parameterlab/MASEval: Multi-Agent LLM Evaluation](https://github.com/parameterlab/MASEval)
python-version... MASEval is an evaluationlibrarythat provides a unified interface for benchmarking (multi-)agent 
systems.

[A Measurement Study of Model Context Protocol](https://arxiv.org/html/2509.25292v1)
... 6.1 Measurement ofLibraryEcosystems for Supply-Chain Security Assessment

[app.readytensor.ai/publications/accelerate-your-aiml-career-with-open-source-contributions-fUTy90FWorvg](https://a
pp.readytensor.ai/publications/accelerate-your-aiml-career-with-open-source-contributions-fUTy90FWorvg)
JAX Jax is a high-performance numerical computinglibraryfrom Google ( 2018 ), combining NumPy-like syntax with 
automatic differentiation and GPU ...

[The State of Python 2025 - Fancy Hints](https://fancyhints.com/the-state-of-python-2025/)
Many of us (15%) are running on the verylatestreleasedversionof Python, but more likely than not, we’re using 
aversiona year old or older (83 ...

Out: None

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("1.24.0")                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1.24.0

[Step 5: Duration 19.44 seconds| Input tokens: 20,841 | Output tokens: 1,101]


Final answer: 1.24.0


The agent issued one or more search queries, read the returned snippets, and extracted the version number from the text — all without any retrieval code written by us. Contrast this with the RAG notebook where we had to build the retriever manually: here the retrieval step is entirely delegated to the tool and the agent's reasoning loop.

> **When to use search vs RAG**
>
> | | `DuckDuckGoSearchTool` | RAG (custom retriever) |
> |---|---|---|
> | Knowledge source | Live web | Your own documents |
> | Latency | Network round-trip | Local (fast) |
> | Control | None — results vary | Full — you index the corpus |
> | Best for | Current events, open-domain facts | Private data, reproducibility |


## 9  `CodeAgent` vs `ToolCallingAgent`

smolagents provides two distinct agent types that differ in *how* the LLM expresses its actions:

| | `CodeAgent` | `ToolCallingAgent` |
|---|---|---|
| Action format | Python code block | JSON tool-call object |
| Execution | Local Python interpreter runs the code | Framework calls the matching tool directly |
| Flexibility | Can use any Python expression | Constrained to declared tool signatures |
| Verbosity | Higher (code + output) | Lower (structured JSON) |
| Best for | Tasks needing complex logic or loops | Tasks with well-defined, discrete tool calls |

> **CodeAgent executes Python; ToolCallingAgent emits JSON**
>
> With `CodeAgent`, the model writes `result = text_statistics(passage)` and the executor runs it as real Python. With `ToolCallingAgent`, the model emits `{"name": "text_statistics", "arguments": {"text": "..."}}` and the framework dispatches the call. Both patterns implement the same Thought → Action → Observation loop — they differ only in the action encoding.

Let's run the same task with both types and compare:

In [14]:
TASK = "What is 123 multiplied by 456? Use the python_interpreter tool to compute it."

# --- CodeAgent ---
code_agent = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=4
)

result_code = code_agent.run(TASK)
print("CodeAgent answer:", result_code)


You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '```python', and closed with '```'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples u

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 123 multiplied by 456? Use the python_interpreter tool to compute it.                                   │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 123 * 456                                                                                               
  final_answer(result)                                                                                             
                                                                                                                   
  result = 123 * 456                                                                                               
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 56088

[Step 1: Duration 9.30 seconds| Input tokens: 2,233 | Output tokens: 148]

CodeAgent answer: 56088


ToolCallingAgent requires a model that can output JSON. 

In [15]:
from smolagents import ToolCallingAgent, TransformersModel

model_tool = TransformersModel(model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct")

Loading weights: 100%|██████████| 338/338 [00:20<00:00, 16.26it/s]


In [16]:
TASK = "What is 123 multiplied by 456? Use the python_interpreter tool to compute it."

# --- ToolCallingAgent ---
tc_agent = ToolCallingAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model_tool,
    max_steps=4,
)

print(tc_agent.system_prompt)

result_tc = tc_agent.run(TASK)
print("ToolCallingAgent answer:", result_tc)

You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.
To do so, you have been given access to some tools.

The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
This Action/Observation can repeat N times, you should take several steps when needed.

You can use the result of the previous action as input for the next action.
The observation will always be a string: it can represent a file, like "image_1.jpg".
Then you can use it as input for the next action. You can do it for instance as follows:

Observation: "image_1.jpg"

Action:
{
  "name": "image_transformer",
  "arguments": {"image": "image_1.jpg"}
}

To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:
Action:
{
  "name": "final_answer"

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 123 multiplied by 456? Use the python_interpreter tool to compute it.                                   │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'python_interpreter' with arguments: {'code': '123 * 456'}                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Stdout:

Output: 56088

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: 56088                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 56088

Final answer: 56088

[Step 2: Duration 0.61 seconds| Input tokens: 2,668 | Output tokens: 63]

ToolCallingAgent answer: 56088


In [17]:
# Compare the raw model output for the first action step
def first_action_output(agent):
    for step in agent.memory.steps:
        if isinstance(step, ActionStep) and step.model_output:
            return step.model_output.strip()
    return "(no action step found)"

print("=== CodeAgent — model output (first action) ===")
print(first_action_output(code_agent))

print("\n=== ToolCallingAgent — model output (first action) ===")
print(first_action_output(tc_agent))

=== CodeAgent — model output (first action) ===
The user wants me to compute 123 multiplied by 456 using the python_interpreter tool. This is a simple multiplication operation.

Let me write the code to compute this:

```python
result = 123 * 456
final_answer(result)
```

This is straightforward - I just need to multiply these two numbers and return the result.
</think>

Thought: I need to compute the multiplication of 123 and 456 using the python_interpreter tool. This is a simple arithmetic operation that I can perform directly in Python code.

```python
result = 123 * 456
final_answer(result)
```

=== ToolCallingAgent — model output (first action) ===
```json
{
  "name": "python_interpreter",
  "arguments": {
    "code": "123 * 456"
  }
}
```


The `CodeAgent` output contains a Python code block (`` ```py ... ``` ``). The `ToolCallingAgent` output contains a structured tool-call (JSON or a tagged block). Both arrive at the same final answer via different internal representations.

## 10  Inspecting the Agent's Memory

After a run, `agent.memory.steps` holds the full execution history as a list of typed step objects:

| Step type | When it appears |
|---|---|
| `SystemPromptStep` | Always first — contains the injected system prompt |
| `TaskStep` | The user's task |
| `ActionStep` | Each LLM reasoning + tool-execution cycle |
| `PlanningStep` | When `planning_interval` is set (optional planning phase) |
| `FinalAnswerStep` | The terminal step with the final output |

In [18]:
# Re-use code_agent from the previous section
print(f"Total steps in memory: {len(code_agent.memory.steps)}\n")

for i, step in enumerate(code_agent.memory.steps):
    step_type = type(step).__name__
    print(f"[{i}] {step_type}")

    if isinstance(step, SystemPromptStep):
        # Show just the first line of the system prompt
        print("    ", step.system_prompt.splitlines()[0][:80])

    elif isinstance(step, TaskStep):
        print("    Task:", step.task[:80])

    elif isinstance(step, ActionStep):
        code_snippet = (step.code_action or "").strip()[:80]
        obs_snippet  = str(step.observations or "")[:60]
        print(f"    Code  : {code_snippet}")
        print(f"    Obs   : {obs_snippet}")
        print(f"    Tokens: {step.token_usage}")

    elif isinstance(step, FinalAnswerStep):
        print("    Output:", str(step.output)[:80])

Total steps in memory: 2

[0] TaskStep
    Task: What is 123 multiplied by 456? Use the python_interpreter tool to compute it.
[1] ActionStep
    Code  : result = 123 * 456
final_answer(result)

result = 123 * 456
final_answer(result)
    Obs   : Execution logs:
Last output from code snippet:
56088
    Tokens: TokenUsage(input_tokens=2233, output_tokens=148, total_tokens=2381)


The memory object is central to agent debugging. By inspecting `ActionStep.code_action` (for `CodeAgent`) or `ActionStep.tool_calls` (for `ToolCallingAgent`) you can see exactly what the model decided to do and why. `token_usage` lets you track cost per step.

## 11  Multi-step Reasoning

Some tasks cannot be solved in a single action — they require computing an intermediate value and then using it in a follow-up calculation. Here we pose such a task explicitly and watch the agent work through it over multiple steps.

In [19]:
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature value in degrees Celsius.
    """
    return celsius * 9 / 5 + 32


@tool
def body_temp_status(fahrenheit: float) -> str:
    """Classify a body temperature given in Fahrenheit as normal, fever, or hypothermia.

    Normal body temperature is 97 – 99 °F. Above 100.4 °F is fever.
    Below 95 °F is hypothermia.

    Args:
        fahrenheit: Body temperature in degrees Fahrenheit.
    """
    if fahrenheit > 100.4:
        return "fever"
    elif fahrenheit < 95.0:
        return "hypothermia"
    else:
        return "normal"


agent_multistep = CodeAgent(
    tools=[celsius_to_fahrenheit, body_temp_status, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=8,
)

result = agent_multistep.run(
    "A patient's temperature is 38.9 °C. "
    "Convert it to Fahrenheit, then classify the result."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A patient's temperature is 38.9 °C. Convert it to Fahrenheit, then classify the result.                         │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  celsius_temp = 38.9                                                                                              
  fahrenheit_temp = celsius_to_fahrenheit(celsius_temp)                                                            
  print(f"Temperature in Fahrenheit: {fahrenheit_temp}")                                                           
  status = body_temp_status(fahrenheit_temp)                                                                       
  print(f"Body temperature status: {status}")                                                                      
  final_answer(f"{fahrenheit_temp} °F - {status}")                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Temperature in Fahrenheit: 102.02
Body temperature status: fever

Final answer: 102.02 °F - fever

[Step 1: Duration 13.16 seconds| Input tokens: 2,254 | Output tokens: 217]


Final answer: 102.02 °F - fever


In [26]:
# Print a compact step-by-step trace
action_steps = [s for s in agent_multistep.memory.steps if isinstance(s, ActionStep)]
print(f"Number of action steps: {len(action_steps)}\n")

for step in action_steps:
    print(f"--- Step {step.step_number} ---")
    print("Code     :", (step.code_action or "").strip()[:120])
    print("Obs      :", str(step.observations or "")[:80])
    print()

Number of action steps: 1

--- Step 1 ---
Code     : celsius_temp = 38.9
fahrenheit_temp = celsius_to_fahrenheit(celsius_temp)
print(f"Temperature in Fahrenheit: {fahrenheit
Obs      : Execution logs:
Temperature in Fahrenheit: 102.02
Body temperature status: fever



The trace shows the two-step chain: the agent first called `celsius_to_fahrenheit` with 38.9, obtained the Fahrenheit value, then passed that value to `body_temp_status`. The intermediate result was preserved in the agent's context across steps — this is the key advantage over a single-turn prompt.

## 12  Advanced: Multi-Agent Systems

For complex workflows it is useful to decompose the problem across **specialised sub-agents** coordinated by an **orchestrator**. smolagents supports this via the `managed_agents` parameter: sub-agents appear as callable tools to the orchestrator, which delegates subtasks to them.

```
Orchestrator (CodeAgent)
  ├── math_agent   ← specialised for arithmetic
  └── text_agent   ← specialised for text analysis
```

Each sub-agent has a `name` and `description` that the orchestrator reads — just like tool descriptions.

In [20]:
# Sub-agent 1: arithmetic specialist
math_agent = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="math_agent",
    description=(
        "Expert at mathematical calculations. "
        "Give it a concrete arithmetic problem and it returns the numerical result."
    ),
    max_steps=4,
    verbosity_level=0,
)

# Sub-agent 2: text analysis specialist
text_agent = CodeAgent(
    tools=[text_statistics, word_frequency, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="text_agent",
    description=(
        "Expert at text analysis. "
        "Give it a text passage and a question about its statistics or word frequency."
    ),
    max_steps=4,
    verbosity_level=0,
)

# Orchestrator: delegates to sub-agents
orchestrator = CodeAgent(
    tools=[FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    managed_agents=[math_agent, text_agent],   # Sub-agents become callable tools
    max_steps=6,
)

result = orchestrator.run(
    "I have a text passage below. First, use the text agent to count its words. "
    "Then use the math agent to compute the square root of that word count. "
    "Report both numbers.\n\n"
    + corpus   # reuse the corpus variable defined in section 7
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I have a text passage below. First, use the text agent to count its words. Then use the math agent to compute   │
│ the square root of that word count. Report both numbers.                                                        │
│                                                                                                                 │
│ Natural language processing enables computers to understand human language. Language models learn statistical   │
│ patterns from large language corpora. Understanding language structure is fundamental to natural language       │
│ processing.                                                                                                     │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  word_count_result = text_agent(task="Count the number of words in the following text passage: Natural language   
  processing enables computers to understand human language. Language models learn statistical patterns from       
  large language corpora. Understanding language structure is fundamental to natural language processing.",        
  additional_args={"text": "Natural language processing enables computers to understand human language. Language   
  models learn statistical patterns from large language corpora. Understanding language structure is fundamental   
  to natural language processing."})                                                                               
  print(f"Word count result: {word_count_result}")                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"Word count: 27, Square root of word count: 5.196152422706632")                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Word count: 27, Square root of word count: 5.196152422706632

[Step 4: Duration 10.96 seconds| Input tokens: 12,362 | Output tokens: 1,020]


Final answer: Word count: 27, Square root of word count: 5.196152422706632


The orchestrator treated each sub-agent as a tool, called them in the required order, and combined their outputs. This pattern scales naturally: you can add more specialised sub-agents (search, database, image analysis) without complicating the orchestrator's reasoning — it only needs to know each agent's name and description.

**Step callbacks** (another advanced feature) let you attach side-effect functions that fire after every step. They are useful for logging, early stopping, or live dashboards:

```python
def log_step(step):
    if isinstance(step, ActionStep):
        print(f"[callback] step {step.step_number} — tokens used: {step.token_usage}")

agent = CodeAgent(
    tools=[...], model=model,
    step_callbacks=[log_step],
)
```

## Summary

| Topic | Key takeaway |
|---|---|
| Agent loop | Observe → Think → Act, repeated until `final_answer` is called |
| `TransformersModel` | Wraps any HuggingFace model; pass `BitsAndBytesConfig` via `model_kwargs` |
| `CodeAgent` | LLM writes Python; local executor runs it; very flexible |
| `ToolCallingAgent` | LLM emits JSON tool calls; cleaner for well-defined tool APIs |
| Tool descriptions | The model reads the docstring — precise descriptions = reliable tool selection |
| `@tool` decorator | Turn any annotated function into a first-class tool in one line |
| Tool composition | Pass multiple tools; the agent chains them autonomously |
| Agent memory | `agent.memory.steps` — inspect every LLM decision and observation |
| `max_steps` | Hard cap on loop iterations; catch `AgentMaxStepsError` for graceful failure |
| Multi-agent | Sub-agents with `name` + `description` become callable tools for an orchestrator |

### Suggested next steps

- **Web-enabled agents** — add `DuckDuckGoSearchTool` or `WikipediaSearchTool` and build a research assistant.
- **RAG + agents** — combine the retriever from the previous notebook with a custom `@tool` so the agent can query your knowledge base.
- **Planning** — set `planning_interval=2` on a long task and inspect `PlanningStep` objects to see how the agent revises its plan.
- **Custom executors** — try `executor_type="docker"` for isolated code execution in untrusted scenarios.
- **Evaluation** — use the smolagents benchmark helpers to measure task success rate across a dataset of tasks.

## References

* smolagents documentation — https://huggingface.co/docs/smolagents
* smolagents GitHub — https://github.com/huggingface/smolagents
* ReAct: Synergising Reasoning and Acting (Yao et al., 2022) — https://arxiv.org/abs/2210.03629
* Qwen2.5 model card — https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
* QLoRA / bitsandbytes quantisation — https://arxiv.org/abs/2305.14314